# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from joblib import dump

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneOut, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample

plt.style.use('seaborn-v0_8-darkgrid')

def locate_project_paths() -> Tuple[Path, Path]:
    """Return project root and processed data directory by walking up from CWD."""
    start = Path.cwd()
    for root in [start, *start.parents]:
        candidate = root / 'data' / 'processed'
        if candidate.exists():
            return root, candidate
    raise FileNotFoundError('Unable to locate data/processed directory from current path')

PROJECT_ROOT, PROCESSED_DIR = locate_project_paths()
IMAGES_DIR = PROJECT_ROOT / 'images'
RESULTS_DIR = PROJECT_ROOT / 'results'
ARTIFACTS_DIR = PROJECT_ROOT / 'models' / 'nested_loso_artifacts'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = 'VO2peak_abs'
TIME_WINDOW = (300.0, 1000.0)
WR_LEVELS = (30, 80)
MIN_SAMPLES_PER_LEVEL = 20
MIN_VALID_FRACTION = 0.6

TIME_COLUMN_CANDIDATES = ('Time', 'time', 't')
WORKRATE_COLUMN_CANDIDATES = ('WorkRate', 'WR', 'work_rate')
EXCLUDED_COLUMNS = {'Time', 'time', 't', 'WorkRate', 'workrate', 'WR', 'wr', 'work_rate'}

In [ ]:
GAS_EXCHANGE_VENTILATION = [
    'VO2pulm', 'VO2_movMean', 'VCO2', 'FO2et', 'FCO2et', 'Vent', 'VT', 'BF'
]
HR_ONLY = ['HR']
HEMODYNAMICS = [
    'SV', 'CO', 'dPdt', 'SPTI', 'RPP', 'DPTI', 'DPTISPTI', 'LVET', 'Cwk',
    'Rp', 'TPR', 'fiSYS', 'fiMAP', 'fiDIA', 'sBP', 'mBP', 'dBP'
]
NIRS = ['absO2Hb_Rx1', 'absHHb_Rx1', 'TSI_Rx1_Tx1Tx2Tx3', 'TSI_FF_Rx1_Tx1Tx2Tx3']
ALL_SIGNALS = sorted({*HR_ONLY, *GAS_EXCHANGE_VENTILATION, *HEMODYNAMICS, *NIRS})
NO_GAS_EXCHANGE = sorted(set(ALL_SIGNALS) - set(GAS_EXCHANGE_VENTILATION))
HEMO_PLUS_NIRS = sorted(set(HEMODYNAMICS + NIRS))
SENSOR_GROUPS: Dict[str, Sequence[str]] = {
    'all_signals': ALL_SIGNALS,
    'no_gas_exchange': NO_GAS_EXCHANGE,
    'hemo_plus_nirs': HEMO_PLUS_NIRS,
    'nirs': NIRS,
}
GROUPS_TO_EVALUATE = ('all_signals', 'no_gas_exchange', 'hemo_plus_nirs', 'nirs')

In [ ]:
def detect_column(columns: Iterable[str], candidates: Sequence[str]) -> str:
    """Return first matching column from candidates; raise if none found."""
    lower_map = {col.lower(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(f'Candidate columns {candidates} not present')

def list_subject_ids(processed_dir: Path) -> List[str]:
    """List subject identifiers that have both data and target CSVs."""
    subjects: List[str] = []
    for data_path in sorted(processed_dir.glob('CT_*_data.csv')):
        subject_id = data_path.stem.rsplit('_', 1)[0]
        target_path = processed_dir / f'{subject_id}_targets.csv'
        if target_path.exists():
            subjects.append(subject_id)
    if not subjects:
        raise FileNotFoundError('No subject pairs found in processed directory')
    return subjects

def load_subject(subject_id: str) -> Tuple[pd.DataFrame, float]:
    """Load time-series dataframe and scalar VO2peak_abs target for a subject."""
    data_path = PROCESSED_DIR / f'{subject_id}_data.csv'
    target_path = PROCESSED_DIR / f'{subject_id}_targets.csv'
    if not data_path.exists() or not target_path.exists():
        raise FileNotFoundError(f'Missing data or target file for {subject_id}')
    data_df = pd.read_csv(data_path)
    target_df = pd.read_csv(target_path)
    if TARGET_COLUMN not in target_df.columns:
        raise KeyError(f'Target column {TARGET_COLUMN} missing in {target_path.name}')
    return data_df, float(target_df[TARGET_COLUMN].iloc[0])

In [ ]:
def slice_prbs_window(df: pd.DataFrame, time_col: str, window: Tuple[float, float]) -> pd.DataFrame:
    """Restrict to the PRBS stimulation window (300-1000 s)."""
    start, stop = window
    window_df = df[(df[time_col] >= start) & (df[time_col] <= stop)].copy()
    if window_df.empty:
        raise ValueError(f'No samples inside window {window}')
    return window_df

def select_numeric_signals(df: pd.DataFrame, candidates: Sequence[str]) -> List[str]:
    """Filter candidate signals to numeric columns present in dataframe."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return [col for col in candidates if col in numeric_cols]

def extract_wr_conditioned_features(
    df: pd.DataFrame,
    signal_columns: Sequence[str],
    wr_column: str,
    min_samples: int,
) -> Dict[str, float]:
    """Compute WR-conditioned stats (mean/std at 30 & 80 W) plus delta."""
    features: Dict[str, float] = {}
    lower_wr, upper_wr = sorted(WR_LEVELS)
    for signal in signal_columns:
        mean_lower = np.nan
        mean_upper = np.nan
        for wr in WR_LEVELS:
            wr_values = df.loc[df[wr_column] == wr, signal].dropna()
            mean_name = f'{signal}_mean_wr{wr}'
            std_name = f'{signal}_std_wr{wr}'
            features[mean_name] = np.nan
            features[std_name] = np.nan
            if wr_values.shape[0] >= min_samples:
                features[mean_name] = float(wr_values.mean())
                features[std_name] = float(wr_values.std(ddof=1))
                if wr == lower_wr:
                    mean_lower = features[mean_name]
                else:
                    mean_upper = features[mean_name]
        delta_name = f'{signal}_mean_delta_wr{upper_wr}_minus_wr{lower_wr}'
        features[delta_name] = (mean_upper - mean_lower) if not np.isnan(mean_lower) and not np.isnan(mean_upper) else np.nan
    return features

In [ ]:
class FoldwiseUnitRescaler(BaseEstimator, TransformerMixin):
    """Divide each feature by train-fold standard deviation (unit-aware fold-wise rescaling)."""

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).astype(float)
        scales = X_df.std(axis=0, ddof=0).replace(0.0, 1.0).fillna(1.0)
        self.scales_ = scales.to_numpy(dtype=float)
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        return X_arr / self.scales_

In [ ]:
def build_feature_table(subject_ids: Sequence[str], group_columns: Sequence[str]) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """Construct feature matrix and aligned targets for a sensor group."""
    feature_rows: List[Dict[str, float]] = []
    targets: Dict[str, float] = {}
    for subject_id in subject_ids:
        data_df, target_value = load_subject(subject_id)
        time_col = detect_column(data_df.columns, TIME_COLUMN_CANDIDATES)
        wr_col = detect_column(data_df.columns, WORKRATE_COLUMN_CANDIDATES)
        prbs_df = slice_prbs_window(data_df, time_col, TIME_WINDOW)
        if group_columns == ALL_SIGNALS:
            candidates = [col for col in prbs_df.columns if col not in EXCLUDED_COLUMNS]
            signal_columns = select_numeric_signals(prbs_df, candidates)
        else:
            signal_columns = select_numeric_signals(prbs_df, group_columns)
        if not signal_columns:
            raise ValueError(f'No valid signals for subject {subject_id}')
        features = extract_wr_conditioned_features(prbs_df, signal_columns, wr_col, MIN_SAMPLES_PER_LEVEL)
        features['subject_id'] = subject_id
        feature_rows.append(features)
        targets[subject_id] = target_value
    feature_df = pd.DataFrame(feature_rows).set_index('subject_id')
    non_null_fraction = feature_df.notna().mean()
    keep_cols = non_null_fraction[non_null_fraction >= MIN_VALID_FRACTION].index.tolist()
    if not keep_cols:
        raise ValueError('Feature set empty after filtering missing values')
    feature_df = feature_df[keep_cols]
    target_series = pd.Series(targets, name=TARGET_COLUMN).loc[feature_df.index]
    return feature_df, target_series, keep_cols

In [ ]:
def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> Dict[str, float]:
    """Summarize regression performance."""
    residuals = predicted - actual
    mse = mean_squared_error(actual, predicted)
    rmse = float(np.sqrt(mse))
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': rmse,
        'R2': r2_score(actual, predicted),
        'MeanResidual': float(residuals.mean()),
        'StdResidual': float(residuals.std(ddof=1)),
    }

def build_pipeline(model_name: str) -> Pipeline:
    """Create leakage-safe preprocessing + model pipeline for tuning."""
    steps = [
        ('unit_rescaler', FoldwiseUnitRescaler()),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
    if model_name == 'Ridge':
        steps.append(('model', Ridge()))
    elif model_name == 'ElasticNet':
        steps.append(('model', ElasticNet(max_iter=50000, tol=1e-4, random_state=42)))
    elif model_name == 'PLS':
        steps.append(('model', PLSRegression()))
    else:
        raise ValueError(f'Unsupported model {model_name}')
    return Pipeline(steps)

def build_param_grid(model_name: str, X_train: pd.DataFrame) -> Dict[str, Sequence]:
    """Return hyperparameter grid tailored to the model and training feature size."""
    if model_name == 'Ridge':
        return {'model__alpha': np.logspace(-3, 2, 12)}
    if model_name == 'ElasticNet':
        return {
            'model__alpha': np.logspace(-3, 1, 12),
            'model__l1_ratio': [0.2, 0.4, 0.6, 0.8],
        }
    if model_name == 'PLS':
        max_components = max(1, min(10, X_train.shape[1], X_train.shape[0]))
        return {'model__n_components': list(range(1, max_components + 1))}
    raise ValueError(f'Unsupported model {model_name}')

def nested_loso_tune_and_predict(
    X: pd.DataFrame,
    y: pd.Series,
    model_name: str,
    artifact_dir: Path = None,
    random_state: int = 42,
    ) -> Tuple[pd.DataFrame, Dict[str, float], pd.DataFrame, pd.DataFrame, pd.Series, pd.DataFrame]:
    """Run LOSO outer validation with inner K-fold tuning to prevent leakage and export fold artifacts."""
    loo = LeaveOneOut()
    prediction_records: List[Dict[str, float]] = []
    param_records: List[Dict[str, float]] = []
    importance_records: List[Dict[str, float]] = []
    artifact_records: List[Dict[str, object]] = []
    feature_names = list(X.columns)

    if artifact_dir is not None:
        artifact_dir.mkdir(parents=True, exist_ok=True)

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline = build_pipeline(model_name)
        param_grid = build_param_grid(model_name, X_train)
        n_splits = min(5, len(X_train))
        best_estimator = pipeline
        best_params = {}

        if n_splits >= 2 and all(len(v) > 0 for v in param_grid.values()):
            inner_cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=inner_cv,
                scoring='neg_mean_squared_error',
                n_jobs=None,
                refit=True,
            )
            search.fit(X_train, y_train)
            best_estimator = search.best_estimator_
            best_params = search.best_params_
        else:
            best_estimator.fit(X_train, y_train)
            best_params = {
                key: val[0] if isinstance(val, (list, tuple, np.ndarray)) else val
                for key, val in param_grid.items()
            }

        predictions = best_estimator.predict(X_test)
        subject_id = X.index[test_idx][0]

        prediction_records.append({
            'subject_id': subject_id,
            'actual_vo2peak': float(y_test.iloc[0]),
            'predicted_vo2peak': float(predictions[0]),
        })

        cleaned_params = {param.replace('model__', ''): value for param, value in best_params.items()}
        cleaned_params['subject_id'] = subject_id
        param_records.append(cleaned_params)

        if model_name in ('Ridge', 'PLS'):
            model = best_estimator.named_steps['model']
            if model_name == 'Ridge':
                values = model.coef_.ravel()
            else:
                values = model.x_loadings_[:, 0].ravel()
            record = {'subject_id': subject_id}
            record.update({name: float(value) for name, value in zip(feature_names, values)})
            importance_records.append(record)

        if artifact_dir is not None:
            artifact_path = artifact_dir / f'{subject_id}_outer_model.joblib'
            artifact_payload = {
                'estimator': best_estimator,
                'model_name': model_name,
                'left_out_subject': subject_id,
                'train_subject_ids': X_train.index.tolist(),
                'feature_columns': feature_names,
                'best_params': {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in cleaned_params.items() if k != 'subject_id'},
                'target_column': TARGET_COLUMN,
            }
            dump(artifact_payload, artifact_path)
            artifact_records.append({
                'subject_id': subject_id,
                'artifact_path': str(artifact_path),
                'n_train_subjects': len(X_train),
            })

    prediction_df = pd.DataFrame(prediction_records).set_index('subject_id').sort_index()
    prediction_df['residual'] = prediction_df['predicted_vo2peak'] - prediction_df['actual_vo2peak']
    prediction_df['mean_vo2'] = (prediction_df['predicted_vo2peak'] + prediction_df['actual_vo2peak']) / 2.0

    metrics = compute_metrics(
        prediction_df['actual_vo2peak'].values,
        prediction_df['predicted_vo2peak'].values
    )
    params_df = pd.DataFrame(param_records).set_index('subject_id').sort_index()
    importance_df = pd.DataFrame(importance_records).set_index('subject_id').sort_index() if importance_records else pd.DataFrame()
    mean_importance = importance_df.mean(axis=0) if not importance_df.empty else pd.Series(dtype=float)
    artifacts_df = pd.DataFrame(artifact_records).set_index('subject_id').sort_index() if artifact_records else pd.DataFrame()

    return prediction_df, metrics, params_df, importance_df, mean_importance, artifacts_df

In [ ]:
def plot_pred_vs_actual(y_true: pd.Series, y_pred: pd.Series, title: str, save_path: Path) -> None:
    """Scatter plot with 1:1 reference highlighting agreement."""
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, color='navy', alpha=0.75)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims, '--', color='gray', label='Ideal 1:1')
    plt.xlabel('Actual VO2peak_abs (L/min)')
    plt.ylabel('Predicted VO2peak_abs (L/min)')
    plt.title(title)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_bland_altman(y_true: pd.Series, y_pred: pd.Series, title: str, save_path: Path) -> None:
    """Bland-Altman visualization communicates bias and agreement limits."""
    mean_values = (y_true + y_pred) / 2.0
    residuals = y_pred - y_true
    mean_residual = residuals.mean()
    std_residual = residuals.std(ddof=1)
    loa = 1.96 * std_residual
    plt.figure(figsize=(6, 4))
    plt.scatter(mean_values, residuals, color='darkorange', alpha=0.75)
    plt.axhline(mean_residual, color='black', linestyle='--', linewidth=1, label='Mean bias')
    plt.axhline(mean_residual + loa, color='red', linestyle='--', linewidth=0.8, label='Upper LOA')
    plt.axhline(mean_residual - loa, color='red', linestyle='--', linewidth=0.8, label='Lower LOA')
    plt.xlabel('Mean VO2peak_abs (L/min)')
    plt.ylabel('Residual (Predicted - Actual)')
    plt.title(title)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_residual_hist(residuals: pd.Series, title: str, save_path: Path) -> None:
    """Histogram summarizing residual distribution to reveal error spread."""
    plt.figure(figsize=(6, 4))
    bins = max(5, min(30, residuals.count()))
    plt.hist(residuals, bins=bins, color='slateblue', alpha=0.8, edgecolor='white')
    plt.axvline(residuals.mean(), color='black', linestyle='--', linewidth=1, label='Mean residual')
    plt.xlabel('Residual (Predicted - Actual)')
    plt.ylabel('Count')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_hyperparam_selection(params_df: pd.DataFrame, model_name: str, title: str, save_path: Path) -> None:
    """Visualize selected hyperparameters across outer folds."""
    plt.figure(figsize=(6, 4))
    if params_df.empty:
        plt.text(0.5, 0.5, 'No hyperparameter data', ha='center', va='center')
    elif model_name == 'ElasticNet':
        plt.scatter(params_df['alpha'], params_df['l1_ratio'], color='purple', alpha=0.8)
        plt.xscale('log')
        plt.xlabel('Alpha')
        plt.ylabel('L1 Ratio')
    else:
        param_name = params_df.columns.drop('subject_id').tolist()[0]
        plt.bar(params_df['subject_id'], params_df[param_name], color='teal', alpha=0.8)
        if model_name in ('Ridge', 'ElasticNet'):
            plt.yscale('log')
        plt.ylabel(param_name)
        plt.xticks(rotation=45, ha='right')
        plt.xlabel('Left-out subject')
    plt.title(title)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def infer_signal_name(feature_name: str, signal_names: Iterable[str]) -> str:
    """Infer the signal prefix used to build a feature name."""
    matches = [signal for signal in signal_names if feature_name.startswith(f"{signal}_")]
    return max(matches, key=len) if matches else ''

def infer_signal_type(feature_name: str, signal_type_map: Dict[str, str]) -> str:
    """Map feature names to signal types using prefix matching."""
    signal = infer_signal_name(feature_name, signal_type_map.keys())
    return signal_type_map.get(signal, 'Other')

def plot_feature_importance(
    importance: pd.Series,
    signal_type_map: Dict[str, str],
    title: str,
    save_path: Path,
    ) -> pd.DataFrame:
    """Plot feature importances with signal-type color coding."""
    if importance.empty:
        plt.figure(figsize=(7, 5))
        plt.text(0.5, 0.5, 'No feature importances available', ha='center', va='center')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()
        return pd.DataFrame()
    df = pd.DataFrame({'feature': importance.index, 'importance': importance.values})
    df['signal_type'] = df['feature'].apply(lambda feat: infer_signal_type(feat, signal_type_map))
    df['is_delta'] = df['feature'].str.contains('delta_wr80_minus_wr30', regex=False)
    df['feature_label'] = df['feature'] + np.where(df['is_delta'], ' (Delta)', '')
    color_map = {
        'Gas Exchange': '#1b9e77',
        'Hemodynamics': '#d95f02',
        'NIRS': '#7570b3',
        'HR': '#e7298a',
        'Other': '#666666',
    }
    df['color'] = df['signal_type'].map(color_map).fillna(color_map['Other'])
    order = df['importance'].abs().sort_values(ascending=True).index
    df_sorted = df.loc[order]
    plt.figure(figsize=(8, 6))
    plt.barh(df_sorted['feature_label'], df_sorted['importance'], color=df_sorted['color'])
    plt.axvline(0.0, color='black', linewidth=0.8)
    plt.xlabel('Average standardized coefficient / loading')
    plt.title(title)
    plt.tight_layout()
    handles = [mpatches.Patch(color=color, label=label) for label, color in color_map.items()]
    plt.legend(handles=handles, title='Signal Type', loc='lower right')
    plt.savefig(save_path, dpi=300)
    plt.close()
    return df_sorted

In [ ]:
def safe_name(value: str) -> str:
    """Return a filesystem-friendly token for filenames."""
    return value.replace(' ', '_').replace('/', '_')

In [ ]:
MODELS = ('Ridge', 'ElasticNet', 'PLS')
subject_ids = list_subject_ids(PROCESSED_DIR)
summary_records: List[Dict[str, object]] = []
B = 1000
SIGNAL_TYPE_MAP = {
    **{signal: 'Gas Exchange' for signal in GAS_EXCHANGE_VENTILATION},
    **{signal: 'Hemodynamics' for signal in HEMODYNAMICS},
    **{signal: 'NIRS' for signal in NIRS},
    **{signal: 'HR' for signal in HR_ONLY},
}

for group_name in GROUPS_TO_EVALUATE:
    columns = SENSOR_GROUPS[group_name]
    try:
        X_group, y_group, retained_features = build_feature_table(subject_ids, columns)
    except ValueError as exc:
        print(f'Skipping {group_name}: {exc}')
        continue

    for model_name in MODELS:
        file_token = f"nested_{safe_name(group_name)}_{safe_name(model_name)}"
        artifact_model_dir = ARTIFACTS_DIR / safe_name(group_name) / safe_name(model_name)

        preds_df, metrics, params_df, fold_importances, mean_importances, artifacts_df = nested_loso_tune_and_predict(
            X_group,
            y_group,
            model_name,
            artifact_dir=artifact_model_dir,
        )

        preds_path = RESULTS_DIR / f'{file_token}_loso_predictions.csv'
        params_path = RESULTS_DIR / f'{file_token}_selected_params.csv'
        artifacts_path = RESULTS_DIR / f'{file_token}_artifacts_manifest.csv'
        preds_df.to_csv(preds_path)
        params_df.to_csv(params_path)
        if not artifacts_df.empty:
            artifacts_df.to_csv(artifacts_path)

        plot_pred_vs_actual(
            preds_df['actual_vo2peak'],
            preds_df['predicted_vo2peak'],
            title=f'{group_name} × {model_name}: Predictions vs Actual',
            save_path=IMAGES_DIR / f'{file_token}_pred_vs_actual.png',
        )
        plot_bland_altman(
            preds_df['actual_vo2peak'],
            preds_df['predicted_vo2peak'],
            title=f'{group_name} × {model_name}: Bland-Altman',
            save_path=IMAGES_DIR / f'{file_token}_bland_altman.png',
        )
        plot_residual_hist(
            preds_df['residual'],
            title=f'{group_name} × {model_name}: Residual Histogram',
            save_path=IMAGES_DIR / f'{file_token}_residual_hist.png',
        )

        params_plot_df = params_df.reset_index()
        plot_hyperparam_selection(
            params_plot_df,
            model_name=model_name,
            title=f'{group_name} × {model_name}: Selected Hyperparameters',
            save_path=IMAGES_DIR / f'{file_token}_hyperparams.png',
        )

        if group_name == 'all_signals' and model_name in ('Ridge', 'PLS') and not mean_importances.empty:
            ranked = mean_importances.reindex(mean_importances.abs().sort_values(ascending=False).index)
            top15 = ranked.head(15)
            plot_feature_importance(
                top15,
                signal_type_map=SIGNAL_TYPE_MAP,
                title=f'{group_name} × {model_name}: Top 15 Feature Importances',
                save_path=IMAGES_DIR / f'{file_token}_top15_feature_importance.png',
            )
            delta_mask = mean_importances.index.to_series().str.contains('delta_wr80_minus_wr30', regex=False)
            delta_ranked = mean_importances[delta_mask].reindex(
                mean_importances[delta_mask].abs().sort_values(ascending=False).index
            )
            if not delta_ranked.empty:
                print(f'{group_name} × {model_name}: top delta features (WR80-WR30)')
                print(delta_ranked.head(5))
            else:
                print(f'{group_name} × {model_name}: no WR80-WR30 delta features retained')

        y_true = preds_df['actual_vo2peak'].to_numpy(dtype=float)
        y_pred = preds_df['predicted_vo2peak'].to_numpy(dtype=float)
        n_samples = len(preds_df)

        bootstrap_mae = np.empty(B)
        bootstrap_rmse = np.empty(B)
        bootstrap_r2 = np.empty(B)
        pairs = np.column_stack([y_true, y_pred])
        for i in range(B):
            sample = resample(pairs, replace=True, n_samples=n_samples)
            sample_true = sample[:, 0]
            sample_pred = sample[:, 1]
            bootstrap_mae[i] = mean_absolute_error(sample_true, sample_pred)
            bootstrap_rmse[i] = float(np.sqrt(mean_squared_error(sample_true, sample_pred)))
            bootstrap_r2[i] = r2_score(sample_true, sample_pred)

        mae_ci = tuple(np.percentile(bootstrap_mae, [2.5, 97.5]))
        rmse_ci = tuple(np.percentile(bootstrap_rmse, [2.5, 97.5]))
        r2_ci = tuple(np.percentile(bootstrap_r2, [2.5, 97.5]))

        metrics_with_ci = {
            'MAE': f"{metrics['MAE']:.3f} [{mae_ci[0]:.3f}, {mae_ci[1]:.3f}]",
            'RMSE': f"{metrics['RMSE']:.3f} [{rmse_ci[0]:.3f}, {rmse_ci[1]:.3f}]",
            'R2': f"{metrics['R2']:.3f} [{r2_ci[0]:.3f}, {r2_ci[1]:.3f}]",
            'MeanResidual': metrics['MeanResidual'],
            'StdResidual': metrics['StdResidual'],
        }

        summary_records.append({
            'Group': group_name,
            'Model': model_name,
            'NSubjects': len(preds_df),
            'NFeatures': len(retained_features),
            'ArtifactCount': int(len(artifacts_df)),
            **metrics_with_ci,
            'mae_ci': mae_ci,
            'rmse_ci': rmse_ci,
            'r2_ci': r2_ci,
        })

summary_df = pd.DataFrame(summary_records).sort_values(['Group', 'Model']).reset_index(drop=True)
summary_path = RESULTS_DIR / 'nested_summary_metrics.csv'
summary_df.to_csv(summary_path, index=False)
summary_df